# Load Spark Environment

In [1]:
# import findspark
import findspark
# initialize findspark with spark directory
findspark.init()
# import pyspark
import pyspark
# create spark contextp
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

In [2]:
# inspect spark session
spark

In [3]:
# import required packages
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# import pyspark functionalities
from pyspark.sql.types import NumericType
from pyspark.ml import Pipeline
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# turn of warnings
import warnings
warnings.filterwarnings("ignore")

# import data

In [24]:
dir = "../Cleaned_data/biweekly_basetable.parquet"

In [35]:
biweekly_data = spark.read.parquet(dir)
biweekly_data = biweekly_data.sort(["ticker", "biweekly_index"])
biweekly_data.show()

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+
|ticker|biweekly_i

In [11]:
biweekly_data.select("industry").distinct().show(truncate=False)

+--------------------------------+
|industry                        |
+--------------------------------+
|Copper                          |
|Silver                          |
|Other Industrial Metals & Mining|
|Other Precious Metals & Mining  |
|Gold                            |
+--------------------------------+



In [26]:
biweekly_data.select("biweekly_period").distinct().count()

46

# Further Preprocessing/ FE

### Price Deviation from previous year

In [18]:
# Price deviation from previous year average (uses all data available from the period before until the period 1 year back)
window_year = Window.partitionBy("ticker").orderBy("period_start").rowsBetween(-26, -1)  # Previous 365 days = 26 periods (excluding current)

# Calculate rolling year average close price
biweekly_data = biweekly_data.withColumn(
    "rolling_year_avg_close", 
    F.avg("avg_close").over(window_year)
)

# Calculate percentage difference from year average
biweekly_data = biweekly_data.withColumn(
    "price_deviation_from_year", 
    (F.col("avg_close") - F.coalesce(F.col("rolling_year_avg_close"), F.col("avg_close"))) / 
    F.coalesce(F.col("rolling_year_avg_close"), F.col("avg_close"))
)

biweekly_data = biweekly_data.drop("rolling_year_avg_close")
biweekly_data.show(5)

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+-------------------+-------+---------+---------------------+------------------------+-----------------------+--------+-------------------+-------------------------+
|ticker|biweekly_index|biweekly_period|            longname|industry|benchmark|period_start|period_end|period_open|period_close|period_high|period_low|         avg_close|       period_return|benchmark_period_return|                beta|    price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_range|total_volume|  avg_daily_volume|max_daily_volume| volume

In [30]:
biweekly_data.write.mode("overwrite").parquet("../Cleaned_data/biweekly_basetable.parquet")

### Lagged Features

In [32]:
window_period = Window.partitionBy("ticker").orderBy("period_start")

# Add features from previous 2-week period
biweekly_data = biweekly_data.withColumn("prev_period_return", F.lag("period_return", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_price_volatility", F.lag("price_volatility", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_avg_daily_volume", F.lag("avg_daily_volume", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_volume_volatility", F.lag("volume_volatility", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_gap_count", F.lag("gap_count", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_exploration_results", F.lag("has_exploration_results", 1).over(window_period))
biweekly_data = biweekly_data.withColumn("prev_property_acquisition", F.lag("has_property_acquisition", 1).over(window_period))

# Add autoregressive features (BE CAUTIOUS WITH DATA LEAKAGE)
# We use data of the last 4 weeks to predict the next 2 weeks, which is a correct setup. 
# You can experiment with even longer lags, it would make sense that a company doesn't announce financings in multiple consecutive periods. 
biweekly_data = biweekly_data.withColumn("current_financing_announcement", F.col("has_financing")) # current period
biweekly_data = biweekly_data.withColumn("prev_financing_announcement", F.lag("has_financing", 1).over(window_period)) # previous period
biweekly_data = biweekly_data.withColumn("second_prev_financing_announcement", F.lag("has_financing", 2).over(window_period)) # second previous period
biweekly_data = biweekly_data.withColumn("third_prev_financing_announcement", F.lag("has_financing", 3).over(window_period)) # previous period


# Step 4: Create comparative features (current vs previous period)
biweekly_data = biweekly_data.withColumn("return_change", 
                                                 F.col("period_return") - F.col("prev_period_return"))
biweekly_data = biweekly_data.withColumn("volatility_change", 
                                                 F.col("price_volatility") / (F.col("prev_price_volatility") + 1e-06))
biweekly_data = biweekly_data.withColumn("volume_change", 
                                                 F.col("avg_daily_volume") / (F.col("prev_avg_daily_volume") + 1e-06))

# Sort by period start date
biweekly_data = biweekly_data.orderBy(["ticker","period_start"])
biweekly_data.show()

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+
|ticker|biweekly_i

In [33]:
biweekly_data = biweekly_data.filter(biweekly_data.financing_next_period.isNotNull())

In [34]:
biweekly_data.write.mode("overwrite").parquet("../Cleaned_data/biweekly_basetable.parquet")

### Adding mining locations

In [28]:
basetable = spark.read.parquet("../Cleaned_data/biweekly_basetable_2.parquet")

In [ ]:
basetable = basetable.drop("locations")

In [30]:
basetable.show()

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+
|ticker|biweekly_i

In [31]:
mining_locations = pd.read_csv("../Data/tickers_mining_locations.csv")

In [34]:
mining_locations.head()

,ticker,locations
0,ARG.TO,[Chile]
1,AYA.TO,[Morocco]
2,CG.TO,"[Canada, Turkey, USA]"
3,CGC.V,[Canada]
4,CKG.V,"[Mexico, USA]"


In [33]:
mining_locations["locations"] = mining_locations["locations"].str.split("; ")

In [35]:
mining_locations_spark = spark.createDataFrame(mining_locations)

In [36]:
mining_locations_spark.show()

+-------+--------------------+
| ticker|           locations|
+-------+--------------------+
| ARG.TO|             [Chile]|
| AYA.TO|           [Morocco]|
|  CG.TO|[Canada, Turkey, ...|
|  CGC.V|            [Canada]|
|  CKG.V|       [Mexico, USA]|
|  DMX.V|            [Sweden]|
|   DV.V|            [Canada]|
|  EDG.V|       [Canada, USA]|
| ELO.TO|[Bolivia, Canada,...|
|    EQX|[Brazil, Canada, ...|
|  ESK.V|            [Canada]|
|  FAS.V|              [Peru]|
| FDY.TO|               [USA]|
| FINX.V|           [Finland]|
|  FNM.V|   [Finland, Sweden]|
| FOM.TO|            [Canada]|
|FURY.TO|            [Canada]|
| FVL.TO|               [USA]|
|  FWZ.V|            [Canada]|
+-------+--------------------+



In [37]:
joined = (
    basetable
    .join(mining_locations_spark, on="ticker", how="left")
)
joined.show()

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+
|ticker|

In [38]:
joined.write.mode("overwrite").parquet("../Cleaned_data/biweekly_basetable.parquet")

### Changing Percentage Change Naming (could cause issues)

In [10]:
basetable = spark.read.parquet("../Cleaned_data/biweekly_basetable.parquet")

In [11]:
joined = basetable.withColumnRenamed(existing="avg_%Chg", new="avg_prChg")
joined.show()

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+
|ticker|

In [12]:
joined.show()

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+---------------------------+----------------------------------+---------------------------------+--------------------+-------------------+------------------+---------+
|ticker|

In [13]:
joined.write.parquet("../Cleaned_data/biweekly_basetable_2.parquet")